# AOSP (PixelOS) Build on Colab Pro

Resumable AOSP builds with persistent caching on Google Drive.

**Setup**: Run cells in order. Subsequent sessions will detect and resume from previous state.


## 1. Mount Google Drive & Install Dependencies

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/root/gdrive')
print(" Google Drive mounted at /root/gdrive")

In [ ]:
# Install AOSP build dependencies
!apt-get update -qq && apt-get install -y -qq \
  git-core gnupg flex bison build-essential zip curl zlib1g-dev \
  gcc-multilib g++-multilib libc6-dev-i386 lib32ncurses5-dev \
  x11-utils libssl-dev libxml2-utils aidl ccache jq git-lfs \
  python3-dev libffi-dev > /dev/null 2>&1

print(" Dependencies installed")
!java -version

In [ ]:
# Install repo tool
!mkdir -p /root/bin
!curl -s https://storage.googleapis.com/git-repo-downloads/repo -o /root/bin/repo
!chmod a+x /root/bin/repo

import os
os.environ['PATH'] = '/root/bin:' + os.environ['PATH']

!repo --version | head -1

## 2. Setup Build Manager

In [ ]:
# Copy build manager script from Drive or create it
import sys
sys.path.insert(0, '/root/aosp')

# Build manager code (paste if not on Drive)
aosp_manager_code = '''
#!/usr/bin/env python3
import os, json, shutil, subprocess, time
from pathlib import Path
from datetime import datetime
from typing import Dict, Optional

class AOSPBuildManager:
    def __init__(self, drive_path, work_path="/root/aosp"):
        self.drive_path = Path(drive_path)
        self.work_path = Path(work_path)
        self.state_file = self.drive_path / "build_state.json"
        self.source_path = self.work_path / "source"
        self.out_path = self.work_path / "out"
        self.ccache_path = self.work_path / ".ccache"
        
    def load_state(self):
        if self.state_file.exists():
            with open(self.state_file) as f:
                return json.load(f)
        return {"last_build": None, "session_count": 0, "build_status": "init"}
    
    def save_state(self, state):
        state["last_sync"] = datetime.now().isoformat()
        with open(self.state_file, 'w') as f:
            json.dump(state, f, indent=2)
        print(f" State saved")
    
    def setup_directories(self):
        self.source_path.mkdir(parents=True, exist_ok=True)
        self.out_path.mkdir(parents=True, exist_ok=True)
        print(f" Directories ready")
    
    def check_space(self):
        stat = shutil.disk_usage(self.work_path)
        total_gb = stat.total / (1024**3)
        free_gb = stat.free / (1024**3)
        used_gb = stat.used / (1024**3)
        print(f"Storage: {used_gb:.1f}GB / {total_gb:.1f}GB used, {free_gb:.1f}GB free")
        return {"total_gb": total_gb, "free_gb": free_gb, "used_gb": used_gb}
    
    def restore_ccache(self):
        ccache_tar = self.drive_path / "ccache.tar.gz"
        if not ccache_tar.exists():
            print("ℹ No ccache found (first build)")
            return False
        print(" Restoring ccache...")
        subprocess.run(f"cd {self.work_path} && tar -xzf {ccache_tar}", shell=True, check=True)
        return True
    
    def backup_ccache(self):
        if not self.ccache_path.exists():
            return False
        print(" Backing up ccache...")
        subprocess.run(f"cd {self.work_path} && tar -czf {self.drive_path}/ccache.tar.gz .ccache/", shell=True, check=True)
        return True
    
    def repo_init(self, manifest_url, branch):
        os.chdir(self.source_path)
        print(f" Initializing repo...")
        cmd = f"repo init -u {manifest_url} -b {branch} --git-lfs"
        return subprocess.run(cmd, shell=True).returncode == 0
    
    def repo_sync(self, jobs=8):
        os.chdir(self.source_path)
        print(f" Syncing repo ({jobs} jobs)...")
        return subprocess.run(f"repo sync -j {jobs} --current-branch", shell=True).returncode == 0
'''

# Save it
with open('/root/aosp/aosp_build_manager.py', 'w') as f:
    f.write(aosp_manager_code)

from aosp_build_manager import AOSPBuildManager
print(" Build manager loaded")

In [ ]:
# Initialize build manager
DRIVE_PATH = "/root/gdrive/MyDrive/AOSP"  # Change if needed
!mkdir -p "$DRIVE_PATH"

manager = AOSPBuildManager(DRIVE_PATH)
manager.setup_directories()

# Check space and load state
space = manager.check_space()
state = manager.load_state()
state['session_count'] = state.get('session_count', 0) + 1

print(f"\n Session #{state['session_count']}")
print(f"Status: {state.get('build_status', 'starting')}")

## 3. Restore from Previous Session (if exists)

In [ ]:
# Try to restore ccache if this is a resumed build
manager.restore_ccache()

# Check if source exists
import os
source_size = sum(f.stat().st_size for f in manager.source_path.rglob('*') if f.is_file()) / (1024**3) if manager.source_path.exists() else 0
if source_size > 1:
    print(f" Source code found ({source_size:.1f}GB)")
    print("\n Ready to build! Jump to cell 5 (Build AOSP) or sync fresh code")
else:
    print("ℹ No source found, proceeding to download (next cells)")

## 4. Initialize & Sync AOSP Source

In [ ]:
# Configure git for repo
!git config --global user.email "aosp@colab.local"
!git config --global user.name "AOSP Builder"
!git config --global color.ui false

print(" Git configured")

In [ ]:
# Initialize repo with PixelOS manifest
MANIFEST_URL = "https://github.com/PixelOS-AOSP/android_manifest.git"
BRANCH = "sixteen-qpr2"

if not manager.repo_init(MANIFEST_URL, BRANCH):
    print(" Repo init failed")
else:
    print(" Repo initialized")

In [ ]:
# Sync repo (this takes time!)
# Use fewer jobs if running into rate limits: repo sync -j 4 --current-branch
import os
os.environ['PATH'] = '/root/bin:' + os.environ['PATH']

if not manager.repo_sync(jobs=8):
    print(" Repo sync failed")
else:
    print(" Source synced")
    manager.check_space()

## 5. Build AOSP

In [ ]:
# Setup build environment
import os
os.chdir(manager.source_path)
os.environ['PATH'] = '/root/bin:' + os.environ['PATH']
os.environ['USE_CCACHE'] = '1'
os.environ['CCACHE_DIR'] = str(manager.ccache_path)
os.environ['CCACHE_MAXSIZE'] = '50G'

print(" Build environment configured")
print(f"CCACHE_DIR: {os.environ['CCACHE_DIR']}")
print(f"PATH: {os.environ['PATH'][:50]}...")

In [ ]:
# Build AOSP
import subprocess
import os

# Set lunch target - modify as needed
# Check available targets with: lunch
LUNCH_TARGET = "cheetah-userdebug"  # Pixel 7 Pro - change to your device
JOBS = 20  # Adjust based on available CPU cores (Colab Pro has 24)

os.chdir(manager.source_path)

build_cmd = f"""
set -e
source build/envsetup.sh > /dev/null
lunch {LUNCH_TARGET}
make -j{JOBS} 2>&1 | tee /root/aosp/build.log
"""

print(f" Building {LUNCH_TARGET}...")
print(f"Using {JOBS} parallel jobs\n")

result = subprocess.run(build_cmd, shell=True, executable='/bin/bash')

if result.returncode == 0:
    print("\n BUILD SUCCESSFUL!")
    state['build_status'] = 'success'
else:
    print("\n Build failed - check logs above")
    state['build_status'] = 'failed'

manager.save_state(state)

## 6. Backup & Upload Results

In [ ]:
# Backup ccache for next session
manager.backup_ccache()

# Check what we have
import shutil
out_size = sum(f.stat().st_size for f in manager.out_path.rglob('*') if f.is_file()) / (1024**3) if manager.out_path.exists() else 0
print(f"\n/out size: {out_size:.1f}GB")

manager.check_space()

In [ ]:
# Compress and upload /out folder
import subprocess
from pathlib import Path
import time

out_dir = manager.out_path
tar_path = manager.drive_path / f"out_{int(time.time())}.tar.gz"

if out_dir.exists():
    print(f" Compressing /out to {tar_path.name}...")
    print("This may take 5-10 minutes...\n")
    
    cmd = f"tar -czf {tar_path} -C {manager.work_path} out/ 2>&1 | grep -E '(^tar|^[0-9]|added)' | tail -5"
    subprocess.run(cmd, shell=True)
    
    compressed_size = tar_path.stat().st_size / (1024**3)
    print(f"\n Uploaded! {tar_path.name} ({compressed_size:.1f}GB)")
    print(f" Location: MyDrive/AOSP/{tar_path.name}")
else:
    print(" No /out folder found")

In [ ]:
# Summary
print(f"""

       BUILD SESSION COMPLETE            


Status: {state.get('build_status', 'unknown')}
Session: #{state['session_count']}

Artifacts on Drive:
  • out_*.tar.gz - Complete build output
  • ccache.tar.gz - Compiler cache (for next build)
  • build_state.json - Session metadata

Next Session:
  1. Run "Mount Drive" cell
  2. Run "Setup" cells
  3. Jump to "Build AOSP" - will auto-restore ccache

Final ROM location in /out:
  • product-userdebug-*.zip
""")

manager.save_state(state)

## 7. Advanced: View Build Logs

In [ ]:
# View last 50 lines of build log
!tail -50 /root/aosp/build.log

In [ ]:
# List final artifacts
!ls -lh /root/aosp/out/target/product/cheetah/*.zip 2>/dev/null || echo "No ROM found - check build status above"

## 8. Cleanup (Optional)

In [ ]:
# Free up space if needed (keeps source & ccache on Drive)
import shutil

# Remove local /out to free space
if manager.out_path.exists():
    shutil.rmtree(manager.out_path)
    print(" Cleared /out folder (backed up to Drive)")

manager.check_space()